In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import os
import re
import csv

In [ ]:
timescaling_datasets = [
    'simulated_binary_6_gens_samplingnoise_seed_2462',
    'simulated_binary_8_gens_samplingnoise_seed_2462',
    'simulated_binary_10_gens_samplingnoise_seed_2462',
    'simulated_binary_12_gens_samplingnoise_seed_2462',
    'simulated_binary_14_gens_samplingnoise_seed_2462',
    'simulated_binary_16_gens_samplingnoise_seed_2462',
    'simulated_binary_13_gens_samplingnoise_seed_2462',
    'simulated_binary_15_gens_samplingnoise_seed_2462',
]

In [ ]:
# Get output files for normal Bonsai runs
outerr_files = {}
metadata_files = {}
final_folders = {}
for dataset in timescaling_datasets:
    path_to_outerr_folder = os.path.join('/scicore/home/nimwegen/degroo0000/Private-bonsai/slurm_runs_pipeline/outerr/simulated_datasets_timescaling', dataset)
    latest_file = max(
        Path(path_to_outerr_folder).glob("core_*.err"),
        key=lambda p: p.stat().st_mtime
    )
    outerr_files[dataset] = os.path.join(path_to_outerr_folder, latest_file)
    final_folders[dataset] = os.path.join('/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/slurm_runs_pipeline/output/simulated_datasets_timescaling', dataset, 'bonsai/final_bonsai_zscore1.0')
    metadata_files[dataset] = os.path.join('/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/slurm_runs_pipeline/output/simulated_datasets_timescaling', dataset, 'bonsai/final_bonsai_zscore1.0/metadata_bonsai.txt')

In [ ]:
peak_mem_pattern = re.compile(
    r"Process (\d+):.*Peak-mem since last:\s*([-+]?\d*\.\d+|\d+)\s*MB"
)
patterns = {
    "loglikelihood": re.compile(
        r"Loglikelihood of inferred tree after reordering children:\s*([-+]?\d*\.\d+|\d+)"
    ),
    "peak_mem": peak_mem_pattern,
    "total_time": re.compile(
        r"Time necessary for the whole calculation was\s*([-+]?\d*\.\d+|\d+)\s*seconds"
    ),
}

results_ds = {}
for dataset in timescaling_datasets:
    path = Path(outerr_files[dataset])
    values = {
        "loglikelihood": None,
        "peak_mem": {},
        "total_time": None,
        "loglikelihood_final": None,
        "n_genes": None,
        "n_cells": None,
        'datafile_size': None,
    }

    # with path.open() as f:
    #     for line in f:
    #         if values["loglikelihood"] is None:
    #             m = patterns["loglikelihood"].search(line)
    #             if m:
    #                 values["loglikelihood"] = float(m.group(1))

    #         if values["peak_mem"] is None:
    #             m = patterns["peak_mem"].search(line)
    #             if m:
    #                 values["peak_mem"] = float(m.group(1))

    #         if values["total_time"] is None:
    #             m = patterns["total_time"].search(line)
    #             if m:
    #                 values["total_time"] = float(m.group(1))

    #         # Early exit once everything is found
    #         if all(v is not None for v in values.values()):
    #             break

    # Read all lines once, then iterate backwards
    with path.open() as f:
        for line in reversed(f.readlines()):
            if values["loglikelihood"] is None:
                m = patterns["loglikelihood"].search(line)
                if m:
                    values["loglikelihood"] = float(m.group(1))

            if len(values["peak_mem"]) < 10:
                m = peak_mem_pattern.search(line)
                if m:
                    proc = int(m.group(1))
                    peak_mem = float(m.group(2))
                    if proc not in values["peak_mem"]:  # only keep the last occurrence
                        values["peak_mem"][proc] = peak_mem

            if values["total_time"] is None:
                m = patterns["total_time"].search(line)
                if m:
                    values["total_time"] = float(m.group(1))

            # Stop as soon as everything we care about is found
            if all(
                values[k] is not None
                for k in ("loglikelihood", "total_time")
            ) and len(values["peak_mem"]) == 10:
                break
    path_to_datafile = os.path.join(final_folders[dataset], 'posterior_ltqs_vertByGene.npy')
    if os.path.exists(path_to_datafile):
        values['datafile_size'] = os.path.getsize(path_to_datafile) / 1e9
    path = Path(metadata_files[dataset])
    if path.exists():
        with path.open() as f:
            reader = csv.DictReader(f)
            row = next(reader)
        values.update({
            "loglikelihood_final": float(row["finalLoglik"]),
            "n_genes": int(row["nGenes"]),
            "n_cells": int(row["nCells"]),
        })
    else:
        print(metadata_files[dataset])

    results_ds[dataset] = values

In [ ]:
for dataset, values in results_ds.items():
    peak_values = values.get("peak_mem", {})
    if peak_values:  # make sure it's not empty
        values["peak_mem_avg_nonzero"] = np.mean([peak_val for ind, peak_val in peak_values.items() if ind != 0])
        values["peak_mem_zero"] = peak_values[0]
    else:
        values["peak_mem_avg_nonzero"] = values["peak_mem_zero"] = None

    # Remove the original dictionary
    del values["peak_mem"]

In [ ]:
df = pd.DataFrame.from_dict(results_ds, orient="index")
df.index.name = "dataset"
df.reset_index().to_csv("timescaling_normal.csv", index=False)

# Do the same for iterative bonsai

In [ ]:
# Get output files for iterative Bonsai runs
cmd_pattern = re.compile(r"_cmd(\d+)_")

outerr_files = {dataset: {} for dataset in timescaling_datasets}
timing_files = {dataset: {} for dataset in timescaling_datasets}
metadata_files = {}
final_folders = {}

for dataset in timescaling_datasets:
    final_folders[dataset] = os.path.join('/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/slurm_runs_pipeline/output/simulated_datasets_timescaling_iterative', dataset, 'bonsai/final_bonsai_zscore1.0_tmpStartnon_refined_tree')
    metadata_files[dataset] = os.path.join('/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/slurm_runs_pipeline/output/simulated_datasets_timescaling_iterative', dataset, 'bonsai/final_bonsai_zscore1.0_tmpStartnon_refined_tree/metadata_bonsai.txt')
    path_to_outerr_folder = Path(
        "/scicore/home/nimwegen/degroo0000/Private-bonsai/slurm_runs_pipeline/outerr/"
        "simulated_datasets_timescaling_iterative"
    ) / dataset

    # Find all .err files recursively
    candidates = path_to_outerr_folder.rglob("*.err")

    # Pick the latest file that matches *_cmdXX_*
    latest_path = None
    latest_mtime = -1
    latest_cmd = None

    for p in candidates:
        m = cmd_pattern.search(p.name)
        if m:
            mtime = p.stat().st_mtime
            cmd_num = int(m.group(1))
            if cmd_num in timing_files[dataset] and mtime < timing_files[dataset][cmd_num]:
                continue
            
            timing_files[dataset][cmd_num] = mtime
            outerr_files[dataset][cmd_num] = str(p)

In [ ]:
peak_mem_pattern = re.compile(
    r"Process (\d+):.*Peak-mem since last:\s*([-+]?\d*\.\d+|\d+)\s*MB"
)
patterns = {
    "loglikelihood": re.compile(
        r"Loglikelihood of inferred tree after reordering children:\s*([-+]?\d*\.\d+|\d+)"
    ),
    "peak_mem": peak_mem_pattern,
    "total_time": re.compile(
        r"Time necessary for the whole calculation was\s*([-+]?\d*\.\d+|\d+)\s*seconds"
    ),
}

n_steps = 4

results_ds = {}
results_per_step_ds = {dataset: {} for dataset in timescaling_datasets if os.path.exists(os.path.exists(metadata_files[dataset]))}
for dataset in timescaling_datasets:
    if not os.path.exists(metadata_files[dataset]):
        continue
    values = {
        "peak_mem": {ind: np.nan for ind in range(10)},
        "total_time": 0.0,
        "loglikelihood_final": None,
        "n_genes": None,
        "n_cells": None,
        "datafile_size": None,
    }
    values.update({"total_time_per_step_{}".format(ind): np.nan for ind in range(n_steps)})
    values["peak_mem_per_step"] = {step_ind: {ind: np.nan for ind in range(10)} for step_ind in range(n_steps)}

    for cmd_num in outerr_files[dataset]:
        path = Path(outerr_files[dataset][cmd_num])
        values_per_step = {
            "peak_mem": {},
            "total_time": None,
        }
    
        # Read all lines once, then iterate backwards
        with path.open() as f:
            for line in reversed(f.readlines()):
                if len(values_per_step["peak_mem"]) < 10:
                    m = peak_mem_pattern.search(line)
                    if m:
                        proc = int(m.group(1))
                        peak_mem = float(m.group(2))
                        if proc not in values_per_step["peak_mem"]:  # only keep the last occurrence
                            values_per_step["peak_mem"][proc] = peak_mem
    
                if values_per_step["total_time"] is None:
                    m = patterns["total_time"].search(line)
                    if m:
                        values_per_step["total_time"] = float(m.group(1))
    
                # Stop as soon as everything we care about is found
                if (values_per_step['total_time'] is not None) and len(values_per_step["peak_mem"]) == 10:
                    break

        results_per_step_ds[dataset] = values_per_step
            
        # Update dataset-values
        for proc, peak_mem in values_per_step["peak_mem"].items():
            if (values['peak_mem'][proc] < peak_mem) or np.isnan(values['peak_mem'][proc]):
                values['peak_mem'][proc] = peak_mem
            values['peak_mem_per_step'][cmd_num][proc] = peak_mem

        values['total_time'] += values_per_step['total_time']
        values['total_time_per_step_{}'.format(cmd_num)] = values_per_step['total_time']

    # Then still get the final loglikelihood
    path_to_datafile = os.path.join(final_folders[dataset], 'posterior_ltqs_vertByGene.npy')
    if os.path.exists(path_to_datafile):
        values['datafile_size'] = os.path.getsize(path_to_datafile) / 1e9
    path = Path(metadata_files[dataset])
    if path.exists():
        with path.open() as f:
            reader = csv.DictReader(f)
            row = next(reader)
        
        values.update({
            "loglikelihood_final": float(row["finalLoglik"]),
            "n_genes": int(row["nGenes"]),
            "n_cells": int(row["nCells"]),
        })
    else:
        print(metadata_files[dataset])

    results_ds[dataset] = values

In [ ]:
for dataset, values in results_ds.items():
    peak_values = values.get("peak_mem", {})
    if peak_values:  # make sure it's not empty
        values["peak_mem_avg_nonzero"] = np.mean([peak_val for ind, peak_val in peak_values.items() if ind != 0])
        values["peak_mem_zero"] = peak_values[0]
    else:
        values["peak_mem_avg_nonzero"] = values["peak_mem_zero"] = None
        print(dataset)

    # Remove the original dictionary
    del values["peak_mem"]

In [ ]:
for dataset, values in results_ds.items():
    peak_values_per_step = values["peak_mem_per_step"]
    for step_ind, peak_values in peak_values_per_step.items():
        nonzero_peak_mems = [peak_val for ind, peak_val in peak_values.items() if ind != 0]
        if np.all(np.isnan(nonzero_peak_mems)):
            values["peak_mem_avg_nonzero_step_{}".format(step_ind)] = np.nan
        else:
            values["peak_mem_avg_nonzero_step_{}".format(step_ind)] = np.nanmean(nonzero_peak_mems)
        values["peak_mem_zero_step_{}".format(step_ind)] = peak_values[0]
        if np.isnan(values["peak_mem_avg_nonzero_step_{}".format(step_ind)]):
            values["peak_mem_avg_nonzero_step_{}".format(step_ind)] = None
        if np.isnan(values["peak_mem_zero_step_{}".format(step_ind)]):
            values["peak_mem_zero_step_{}".format(step_ind)] = None
    
    # Remove the original dictionary
    del values["peak_mem_per_step"]

In [ ]:
df = pd.DataFrame.from_dict(results_ds, orient="index")
df.index.name = "dataset"
df.reset_index().to_csv("timescaling_iterative.csv", index=False)

In [ ]:
results_ds